# Module 13: Distributed Messaging Event Streaming Queues — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/commit_log_stream.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import commit_log_stream

classes = [n for n, o in inspect.getmembers(commit_log_stream, inspect.isclass)
           if o.__module__ == 'commit_log_stream']
functions = [n for n, o in inspect.getmembers(commit_log_stream, inspect.isfunction)
             if o.__module__ == 'commit_log_stream']

print('module   : commit_log_stream')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(commit_log_stream, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Key based partitioning guarantees order

This is the module's own `test_key_based_partitioning_guarantees_order` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from commit_log_stream import (
    ConsumerGroup,
    Topic,
)

topic = Topic("order-events", num_partitions=3)

# Invariant: All events with the same key MUST map to the same partition!
r1 = topic.publish(key="order-991", value={"status": "CREATED"})
r2 = topic.publish(key="order-991", value={"status": "PAID"})
r3 = topic.publish(key="order-991", value={"status": "SHIPPED"})

assert r1.partition_id == r2.partition_id == r3.partition_id
assert r1.offset == 0
assert r2.offset == 1
assert r3.offset == 2

print('PASSED: test_key_based_partitioning_guarantees_order')

## 3. 🔮 Prediction — commit before you run

A consumer crashes after processing a message but before acknowledging it. Predict whether that message is lost, redelivered, or silently dropped.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_consumer_group_partition_assignment_and_rebalance`, which tests exactly this property.


In [ ]:
topic = Topic("clicks", num_partitions=4)
cg = ConsumerGroup(group_id="analytics-service", topic=topic)

# 1. Register Consumer 1 -> owns all 4 partitions
cg.register_member("worker-1")
assert cg.assignments["worker-1"] == [0, 1, 2, 3]

# 2. Register Consumer 2 -> rebalance splits partitions evenly (2 each)
cg.register_member("worker-2")
assert len(cg.assignments["worker-1"]) == 2
assert len(cg.assignments["worker-2"]) == 2
# Combined union covers all 4 partitions
all_assigned = set(cg.assignments["worker-1"] + cg.assignments["worker-2"])
assert all_assigned == {0, 1, 2, 3}

# 3. Consumer 2 leaves -> Consumer 1 reclaims all 4 partitions
cg.leave_member("worker-2")
assert cg.assignments["worker-1"] == [0, 1, 2, 3]

print('PASSED: test_consumer_group_partition_assignment_and_rebalance')

## 4. Measure it: Consumer group fetch and offset commit

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_consumer_group_fetch_and_offset_commit` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

topic = Topic("telemetry", num_partitions=2)
cg = ConsumerGroup(group_id="metrics-writer", topic=topic)
cg.register_member("c1")

# Publish 4 messages
for i in range(4):
    topic.publish(key=f"sensor-{i}", value={"temp": 20 + i})

# Fetch batch 1
batch1 = cg.fetch("c1", max_records_per_partition=10)
assert len(batch1) == 4

# Before commit, fetching again returns the same records (at-least-once guarantee)
batch1_repeat = cg.fetch("c1", max_records_per_partition=10)
assert len(batch1_repeat) == 4

# Commit the batch
cg.commit(batch1)

# Fetching after commit returns 0 new records
batch2 = cg.fetch("c1", max_records_per_partition=10)
assert len(batch2) == 0

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_consumer_group_fetch_and_offset_commit')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(commit_log_stream) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. At-least-once plus idempotent consumers is the achievable guarantee.
2. A commit log is a replayable ordered fact stream, not a work queue.
3. Consumer groups partition work; offsets are what make crash recovery possible.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
